# Composing agents with `as_tool()`

The [agents as tools](./agent-as-tools.ipynb) notebook shows that one agent can call another. This notebook is about the decisions you make once you start doing that.

`Agent.as_tool()` turns an agent into a tool another agent can call. It takes two keywords that look like small configuration details but actually decide what kind of system you are building:

| Keyword | The question it answers |
|---|---|
| `preserve_context` | Should this specialist remember anything between calls? |
| `delegate` | Who writes the final answer, the specialist or the orchestrator? |

Neither has a universally correct value. This notebook runs both settings both ways so you can see what each produces, then gives you a rule for choosing.

## Setup

In [ ]:
import os

from strands import Agent, tool

MODEL_ID = "us.anthropic.claude-sonnet-5"
os.environ.setdefault("AWS_REGION", "us-east-1")

## Wiring: `as_tool()` versus a hand-rolled wrapper

There are two ways to expose an agent as a tool.

The **hand-rolled** way, used in the [agents as tools](./agent-as-tools.ipynb) notebook, wraps the agent in your own `@tool` function:

```python
@tool
def research_assistant(query: str) -> str:
    """Answer research questions."""
    return str(research_agent(query))
```

The **SDK-native** way asks the agent for its own tool:

```python
research_agent.as_tool()
```

Both work. Write the wrapper when you need to control the tool's signature, validate arguments, or post-process the result. Use `as_tool()` when you want the agent exposed as-is, because it derives the tool name and description from the agent and gives you the two keywords below.

In [ ]:
researcher = Agent(
    name="researcher",
    description="Answers factual research questions on any topic.",
    system_prompt="You answer research questions in one or two sentences.",
    model=MODEL_ID,
    callback_handler=None,
)

research_tool = researcher.as_tool()

print(f"tool name       : {research_tool.tool_name}")
print(f"tool description: {research_tool.tool_spec['description']}")

## Decision 1: should this specialist remember?

`preserve_context` controls whether the sub-agent keeps its conversation history between calls.

By default it is `False`, which means the specialist's messages and state are reset to their construction-time values **before every call**. Each invocation starts from the same baseline, no matter what happened previously.

To see this you have to isolate the specialist's memory from the orchestrator's. The orchestrator has a conversation history of its own, and if you simply ask it twice it will answer the second question from its own memory without the specialist needing to recall anything. So the demo below **clears the orchestrator's history between the two calls**, leaving the specialist as the only thing that could possibly remember.

In [ ]:
def ask_twice(preserve_context: bool) -> str:
    """Store a fact, wipe the orchestrator's memory, then ask for the fact back.

    Clearing `orchestrator.messages` is what makes this a fair test: afterwards the
    only place the answer can survive is inside the specialist itself.
    """
    specialist = Agent(
        name="note_keeper",
        description="Stores and recalls short notes for the user.",
        system_prompt=(
            "You keep short notes for the user. If you were not told something, "
            "say you do not have it. Answer in one short sentence."
        ),
        model=MODEL_ID,
        callback_handler=None,
    )
    orchestrator = Agent(
        model=MODEL_ID,
        system_prompt="Use note_keeper for anything about the user's notes. Answer briefly.",
        tools=[specialist.as_tool(preserve_context=preserve_context)],
        callback_handler=None,
    )

    orchestrator("Tell note_keeper to remember that my favourite colour is teal.")
    orchestrator.messages = []  # the orchestrator forgets; only the specialist might not
    return str(orchestrator("Ask note_keeper what my favourite colour is.")).strip()


for preserve in (False, True):
    answer = ask_twice(preserve)
    print(f"preserve_context={preserve}")
    print(f"  recalled 'teal': {'teal' in answer.lower()}")
    print(f"  {answer[:130]}\n")

### Choosing

| | What happens | Reach for it when the specialist is |
|---|---|---|
| `preserve_context=False` (default) | Every call starts from the construction-time baseline | A pure function: a translator, a classifier, a lookup. Two unrelated requests can never contaminate each other |
| `preserve_context=True` | History accumulates across calls | Working a single case over several turns, where forgetting the last exchange makes it useless |

The default is stateless on purpose. A specialist that quietly accumulates history is harder to reason about, because its answer depends on calls made elsewhere in your program. Opt into memory when the role genuinely needs it.

## Decision 2: who writes the final answer?

When a specialist returns, the orchestrator normally reads that result and composes its own reply. `delegate=True` changes that: the specialist's response becomes the final response, and the orchestrator stops.

To make the difference visible, the billing specialist below is told to prefix every reply with `BILLING SAYS:`. If that prefix survives to the final answer, the specialist spoke directly. If it does not, the orchestrator rewrote it.

In [ ]:
def support_desk(delegate: bool) -> Agent:
    """Build a support orchestrator that either relays or paraphrases the billing specialist."""
    billing = Agent(
        name="billing_specialist",
        description="Handles billing, charges and refund questions.",
        system_prompt="You handle billing issues. Always begin your reply with 'BILLING SAYS:'.",
        model=MODEL_ID,
        callback_handler=None,
    )
    return Agent(
        model=MODEL_ID,
        system_prompt="Route billing questions to the billing specialist.",
        tools=[billing.as_tool(delegate=delegate)],
        callback_handler=None,
    )


QUESTION = "I was charged twice for my subscription. Can I get a refund?"

for delegate in (False, True):
    answer = str(support_desk(delegate)(QUESTION)).strip()
    print(f"delegate={delegate}")
    print(f"  specialist answered directly: {answer.startswith('BILLING SAYS:')}")
    print(f"  {answer[:140]}\n")

### Choosing

| | What happens | Reach for it when your orchestrator is |
|---|---|---|
| `delegate=False` (default) | Orchestrator reads the result and writes its own reply | A **synthesizer**, combining several specialists into one answer |
| `delegate=True` | The specialist's reply is returned as-is | A **router**, where the specialist owns the domain and its exact wording should reach the user |

Delegation also saves the extra model call the orchestrator would have spent rewording an answer that was already correct. That matters when the specialist's precision is the point, as with a policy quote, a legal disclaimer, or a diagnosis.

## Putting both decisions to work

A support desk with three specialists. Each answers the two questions differently, and the answers follow from the role rather than from a preference:

| Specialist | Remember? | Owns the answer? | Why |
|---|---|---|---|
| `faq_lookup` | No | No | A lookup should return the same answer to the same question forever |
| `case_handler` | **Yes** | No | It is working one customer's case across several turns |
| `refund_policy` | No | **Yes** | Policy wording is the product; paraphrasing it introduces error |

In [ ]:
faq_lookup = Agent(
    name="faq_lookup",
    description="Answers general questions about the product and account settings.",
    system_prompt="Answer product FAQs in one sentence.",
    model=MODEL_ID,
    callback_handler=None,
)

case_handler = Agent(
    name="case_handler",
    description="Tracks the details of the customer's ongoing support case.",
    system_prompt=(
        "You track one customer's support case. Refer back to details they gave earlier. "
        "Answer in one or two sentences."
    ),
    model=MODEL_ID,
    callback_handler=None,
)

refund_policy = Agent(
    name="refund_policy",
    description="States the official refund policy. Its wording is authoritative.",
    system_prompt=(
        "You state the refund policy exactly as written, beginning with 'POLICY:'. "
        "Never soften or reinterpret it. The policy is:\n"
        "Damaged goods may be returned within 30 days of delivery for a full refund, "
        "including original shipping. Photographic evidence of the damage is required. "
        "Refunds are issued to the original payment method within 5 business days."
    ),
    model=MODEL_ID,
    callback_handler=None,
)

desk = Agent(
    model=MODEL_ID,
    system_prompt=(
        "You are a support desk. Route FAQs to faq_lookup, case details to case_handler, "
        "and any refund policy question to refund_policy."
    ),
    tools=[
        faq_lookup.as_tool(),                              # stateless lookup
        case_handler.as_tool(preserve_context=True),       # remembers the case
        refund_policy.as_tool(delegate=True),              # speaks for itself
    ],
    callback_handler=None,
)

print(desk("My order number is A-4471 and the package arrived damaged."), "\n")
print(desk("What was my order number again?"), "\n")
print(desk("What is your refund policy for damaged goods?"))

## Summary

`as_tool()` exposes an agent as a tool, and its two keywords decide the shape of the system you build.

- **`preserve_context`** answers *should this specialist remember?* Stateless by default, so each call is reproducible. Turn it on when the specialist is working one continuing case.
- **`delegate`** answers *who writes the final answer?* Off by default, so the orchestrator synthesizes. Turn it on when you are routing and the specialist's own wording should reach the user.

A useful check when adding a specialist: name its role in one sentence. If the sentence contains "for this conversation" you probably want `preserve_context=True`. If it contains "authoritative" or "exact", you probably want `delegate=True`.

Next, see [swarm](../11-swarm/swarm.ipynb) and [graph](../12-graph/graph.ipynb) for orchestration patterns where control flow, rather than tool calls, decides which agent runs.